# 시드 문맥 감성 통제 분석

## 목적

`v_2_1_seed_term_context_sentiment.ipynb`에서 만든 seed-only 문맥 감성 변수를 사용해, 긍정 문맥과 부정 문맥이 ESG 등급과 어떻게 관련되는지 더 엄밀하게 확인한다.

핵심 질문은 다음과 같다.

1. 긍정 문맥과 부정 문맥을 서로 통제하면 각각의 방향성이 드러나는가?
2. 문서 길이와 ESG 문장량을 통제해도 긍정/부정 문맥 비중이 ESG 등급과 관련되는가?
3. 긍정과 부정의 순효과(`positive - negative`)가 ESG 등급과 관련되는가?
4. 긍정과 부정이 동시에 높은 기업과 그렇지 않은 기업 사이에 ESG 등급 차이가 있는가?
5. 이 관계가 연도별로도 비슷하게 나타나는가?

## 입력

- `final/v_2_1_seed_term_context_sentiment_analysis.csv`

이 파일은 기업-연도 단위로 시드 단어 포함 문장의 감성, 시드 단어 문맥 감성, ESG 등급을 포함한다.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr, kruskal
import statsmodels.api as sm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

LOCAL_ROOT = Path.cwd()
ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/My Drive/UD_26"),
    LOCAL_ROOT,
    LOCAL_ROOT.parent,
]


def first_existing(candidates, default=None):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return default if default is not None else candidates[0]


ROOT = first_existing(
    [p for p in ROOT_CANDIDATES if (p / "data").exists() or (p / "final").exists()],
    LOCAL_ROOT,
)
FINAL_DIR = ROOT / "final"
INPUT_PATH = first_existing([
    FINAL_DIR / "v_2_1_seed_term_context_sentiment_analysis.csv",
    LOCAL_ROOT / "final" / "v_2_1_seed_term_context_sentiment_analysis.csv",
    LOCAL_ROOT / "v_2_1_seed_term_context_sentiment_analysis.csv",
])

print("ROOT:", ROOT)
print("INPUT_PATH:", INPUT_PATH, "|", "OK" if INPUT_PATH.exists() else "MISSING")

ROOT: /content/drive/MyDrive/UD_26
INPUT_PATH: /content/drive/MyDrive/UD_26/final/v_2_1_seed_term_context_sentiment_analysis.csv | OK


In [2]:
analysis_df = pd.read_csv(INPUT_PATH, dtype={"stock_code": "string"}, encoding="utf-8-sig")
analysis_df["stock_code"] = analysis_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
for col in ["fiscal_year", "esg_year", "esg_grade_num"]:
    if col in analysis_df.columns:
        analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

print("analysis_df:", analysis_df.shape)
print("missing esg_grade_num:", analysis_df["esg_grade_num"].isna().sum())
print("fiscal/esg years:")
display(analysis_df.groupby(["fiscal_year", "esg_year"]).size().rename("n").reset_index())
display(analysis_df.head())

analysis_df: (378, 38)
missing esg_grade_num: 0
fiscal/esg years:


,fiscal_year,esg_year,n
0,2022,2023,126
1,2023,2024,126
2,2024,2025,126


,stock_code,company_name,fiscal_year,esg_year,rcept_no,total_word_count,total_char_count,section_count,seed_sentence_count,positive_seed_sentence_count,negative_seed_sentence_count,neutral_seed_sentence_count,mean_seed_sentence_sentiment,unique_seed_terms_in_sentences,seed_term_context_count,seed_term_occurrence_count,positive_seed_term_context_count,negative_seed_term_context_count,neutral_seed_term_context_count,mean_seed_term_context_sentiment,unique_seed_terms_matched,positive_seed_sentence_share,negative_seed_sentence_share,neutral_seed_sentence_share,positive_seed_term_context_share,negative_seed_term_context_share,seed_sentence_per_1000_words,seed_term_context_per_1000_words,seed_term_occurrence_per_1000_words,industry,esg_grade,e_grade,s_grade,g_grade,esg_grade_num,e_grade_num,s_grade_num,g_grade_num
0,000020,동화약품,2022,2023,20230315001100,7808,37953,3,35,2,2,31,0.018317,14,68,118,2,3,63,-0.000832,14,0.057143,0.057143,0.885714,0.029412,0.044118,4.482582,8.709016,15.112705,NaN,C,C,B,C,1,1,2,1
1,000020,동화약품,2023,2024,20240319000652,8065,38467,3,35,2,1,32,0.036592,13,67,114,2,2,63,0.008703,13,0.057143,0.028571,0.914286,0.029851,0.029851,4.339740,8.307502,14.135152,NaN,C,B,B,C,1,2,2,1
2,000020,동화약품,2024,2025,20250318000739,8097,39259,3,37,2,0,35,0.053468,14,68,115,2,0,66,0.029093,14,0.054054,0.000000,0.945946,0.029412,0.000000,4.569594,8.398172,14.202791,NaN,C,B,C,C,1,2,1,1
3,000040,KR모터스,2022,2023,20230322001182,4201,20136,3,11,1,0,10,0.090888,10,18,38,1,0,17,0.055543,10,0.090909,0.000000,0.909091,0.055556,0.000000,2.618424,4.284694,9.045465,NaN,D,D,D,D,0,0,0,0
4,000040,KR모터스,2023,2024,20240321002062,4888,22402,3,11,0,1,10,-0.047889,11,16,38,0,1,15,-0.032923,11,0.000000,0.090909,0.909091,0.000000,0.062500,2.250409,3.273322,7.774141,NaN,D,D,D,D,0,0,0,0


In [3]:
# Derived variables for testing whether positive and negative contexts offset each other.

analysis_df = analysis_df.copy()

analysis_df["net_seed_sentence_count"] = (
    analysis_df["positive_seed_sentence_count"] - analysis_df["negative_seed_sentence_count"]
)
analysis_df["net_seed_sentence_share"] = (
    analysis_df["positive_seed_sentence_share"] - analysis_df["negative_seed_sentence_share"]
)
analysis_df["net_seed_term_context_count"] = (
    analysis_df["positive_seed_term_context_count"] - analysis_df["negative_seed_term_context_count"]
)
analysis_df["net_seed_term_context_share"] = (
    analysis_df["positive_seed_term_context_share"] - analysis_df["negative_seed_term_context_share"]
)

# Volume-adjusted intensity variables.
analysis_df["positive_seed_sentence_per_1000_words"] = (
    1000 * analysis_df["positive_seed_sentence_count"] / analysis_df["total_word_count"].replace(0, np.nan)
)
analysis_df["negative_seed_sentence_per_1000_words"] = (
    1000 * analysis_df["negative_seed_sentence_count"] / analysis_df["total_word_count"].replace(0, np.nan)
)
analysis_df["positive_seed_term_context_per_1000_words"] = (
    1000 * analysis_df["positive_seed_term_context_count"] / analysis_df["total_word_count"].replace(0, np.nan)
)
analysis_df["negative_seed_term_context_per_1000_words"] = (
    1000 * analysis_df["negative_seed_term_context_count"] / analysis_df["total_word_count"].replace(0, np.nan)
)

# Centered terms for interaction models.
for col in [
    "positive_seed_sentence_share", "negative_seed_sentence_share",
    "positive_seed_term_context_share", "negative_seed_term_context_share",
    "positive_seed_sentence_count", "negative_seed_sentence_count",
]:
    analysis_df[f"c_{col}"] = analysis_df[col] - analysis_df[col].mean(skipna=True)

analysis_df["sentence_share_interaction"] = (
    analysis_df["c_positive_seed_sentence_share"] * analysis_df["c_negative_seed_sentence_share"]
)
analysis_df["term_context_share_interaction"] = (
    analysis_df["c_positive_seed_term_context_share"] * analysis_df["c_negative_seed_term_context_share"]
)
analysis_df["sentence_count_interaction"] = (
    analysis_df["c_positive_seed_sentence_count"] * analysis_df["c_negative_seed_sentence_count"]
)

summary_cols = [
    "positive_seed_sentence_count", "negative_seed_sentence_count", "net_seed_sentence_count",
    "positive_seed_sentence_share", "negative_seed_sentence_share", "net_seed_sentence_share",
    "positive_seed_term_context_count", "negative_seed_term_context_count", "net_seed_term_context_count",
    "unique_seed_terms_matched", "total_word_count", "esg_grade_num",
]
display(analysis_df[summary_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
positive_seed_sentence_count,378.0,9.214286,14.414262,0.000000,2.000000,5.000000,10.000000,118.000000
negative_seed_sentence_count,378.0,2.775132,4.139442,0.000000,1.000000,2.000000,3.000000,50.000000
net_seed_sentence_count,378.0,6.439153,11.513699,-7.000000,1.000000,3.000000,8.000000,83.000000
positive_seed_sentence_share,378.0,0.083938,0.065005,0.000000,0.036785,0.067606,0.125910,0.326241
negative_seed_sentence_share,378.0,0.029213,0.023426,0.000000,0.012987,0.026316,0.040816,0.142857
net_seed_sentence_share,378.0,0.054725,0.066650,-0.142857,0.007663,0.039088,0.101271,0.295918
positive_seed_term_context_count,378.0,12.042328,19.904368,0.000000,2.000000,6.000000,12.000000,164.000000
negative_seed_term_context_count,378.0,4.111111,9.982655,0.000000,1.000000,2.000000,4.000000,126.000000
net_seed_term_context_count,378.0,7.931217,14.280834,-9.000000,0.000000,3.000000,9.000000,96.000000
unique_seed_terms_matched,378.0,18.666667,6.338799,8.000000,14.000000,18.000000,22.000000,37.000000


In [4]:
def spearman_for_feature(data, y_col, x_col):
    tmp = data[[y_col, x_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 3 or tmp[x_col].nunique() < 2 or tmp[y_col].nunique() < 2:
        return len(tmp), np.nan, np.nan
    rho, p_value = spearmanr(tmp[x_col], tmp[y_col])
    return len(tmp), float(rho), float(p_value)


SPEARMAN_FEATURES = [
    "positive_seed_sentence_count",
    "negative_seed_sentence_count",
    "net_seed_sentence_count",
    "positive_seed_sentence_share",
    "negative_seed_sentence_share",
    "net_seed_sentence_share",
    "mean_seed_sentence_sentiment",
    "positive_seed_term_context_count",
    "negative_seed_term_context_count",
    "net_seed_term_context_count",
    "positive_seed_term_context_share",
    "negative_seed_term_context_share",
    "net_seed_term_context_share",
    "mean_seed_term_context_sentiment",
    "unique_seed_terms_matched",
    "seed_sentence_count",
    "seed_term_context_count",
    "seed_term_occurrence_count",
    "total_word_count",
]

spearman_rows = []
for feature in SPEARMAN_FEATURES:
    if feature not in analysis_df.columns:
        continue
    n, rho, p_value = spearman_for_feature(analysis_df, "esg_grade_num", feature)
    spearman_rows.append({"feature": feature, "n": n, "spearman_rho": rho, "p_value": p_value})

spearman_df = pd.DataFrame(spearman_rows).sort_values("spearman_rho", ascending=False).reset_index(drop=True)
display(spearman_df)

,feature,n,spearman_rho,p_value
0,seed_term_occurrence_count,378,0.721106,6.753280e-62
1,seed_term_context_count,378,0.679714,1.464781e-52
2,total_word_count,378,0.653063,2.529959e-47
3,seed_sentence_count,378,0.609911,6.987739e-40
4,unique_seed_terms_matched,378,0.605798,3.125205e-39
5,positive_seed_sentence_count,378,0.559938,1.406746e-32
6,positive_seed_term_context_count,378,0.551829,1.658599e-31
7,net_seed_sentence_count,378,0.506734,4.744407e-26
8,net_seed_term_context_count,378,0.488380,4.721071e-24
9,positive_seed_sentence_share,378,0.381657,1.486778e-14


In [5]:
def zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std


def run_robust_ols(data, y_col, x_cols, min_n=3):
    required_cols = [y_col] + x_cols
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        return None, {"skip_reason": f"missing columns: {missing_cols}", "n": 0}
    reg_df = data[required_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(reg_df) < min_n:
        return None, {"skip_reason": f"too few complete rows after dropna: {len(reg_df)}", "n": int(len(reg_df))}
    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(zscore)
    X = sm.add_constant(X, has_constant="add")
    model = sm.OLS(y, X).fit(cov_type="HC3")
    return model, {"skip_reason": "", "n": int(model.nobs)}


def run_model_table(data, model_specs, y_col="esg_grade_num"):
    rows = []
    for model_name, x_cols in model_specs.items():
        model, info = run_robust_ols(data, y_col, x_cols)
        if model is None:
            rows.append({"model": model_name, "variable": "SKIPPED", **info})
            continue
        for variable in model.params.index:
            rows.append({
                "model": model_name,
                "variable": variable,
                "coef": float(model.params[variable]),
                "std_err": float(model.bse[variable]),
                "p_value": float(model.pvalues[variable]),
                "r2": float(model.rsquared),
                "n": int(model.nobs),
                "skip_reason": "",
            })
    return pd.DataFrame(rows)

In [6]:
# Main models: positive and negative contexts are included together.
# This tests whether one effect is hidden by the other when examined separately.

MODEL_SPECS = {
    "M1_pos_neg_sentence_counts": [
        "positive_seed_sentence_count", "negative_seed_sentence_count",
    ],
    "M2_pos_neg_sentence_counts_length": [
        "positive_seed_sentence_count", "negative_seed_sentence_count", "total_word_count",
    ],
    "M3_pos_neg_sentence_shares_volume": [
        "positive_seed_sentence_share", "negative_seed_sentence_share", "seed_sentence_count", "total_word_count",
    ],
    "M4_net_sentence_count_length": [
        "net_seed_sentence_count", "seed_sentence_count", "total_word_count",
    ],
    "M5_net_sentence_share_volume": [
        "net_seed_sentence_share", "seed_sentence_count", "total_word_count",
    ],
    "M6_pos_neg_term_context_counts": [
        "positive_seed_term_context_count", "negative_seed_term_context_count", "unique_seed_terms_matched",
    ],
    "M7_pos_neg_term_context_counts_length": [
        "positive_seed_term_context_count", "negative_seed_term_context_count", "unique_seed_terms_matched", "total_word_count",
    ],
    "M8_net_term_context_count_length": [
        "net_seed_term_context_count", "unique_seed_terms_matched", "total_word_count",
    ],
    "M9_sentence_share_interaction": [
        "positive_seed_sentence_share", "negative_seed_sentence_share", "sentence_share_interaction", "seed_sentence_count", "total_word_count",
    ],
    "M10_term_context_share_interaction": [
        "positive_seed_term_context_share", "negative_seed_term_context_share", "term_context_share_interaction", "seed_term_context_count", "total_word_count",
    ],
}

ols_df = run_model_table(analysis_df, MODEL_SPECS)
display(ols_df)

# Compact view for non-constant variables only.
compact_ols_df = ols_df[ols_df["variable"].ne("const")].sort_values(["model", "p_value"])
display(compact_ols_df)

,model,variable,coef,std_err,p_value,r2,n,skip_reason
0,M1_pos_neg_sentence_counts,const,2.619048,0.077922,1.160465e-247,0.131508,378,
1,M1_pos_neg_sentence_counts,positive_seed_sentence_count,0.678335,0.121480,2.351748e-08,0.131508,378,
2,M1_pos_neg_sentence_counts,negative_seed_sentence_count,-0.129014,0.121349,2.877071e-01,0.131508,378,
3,M2_pos_neg_sentence_counts_length,const,2.619048,0.072627,9.097125e-285,0.247181,378,
4,M2_pos_neg_sentence_counts_length,positive_seed_sentence_count,0.238031,0.126036,5.894600e-02,0.247181,378,
5,M2_pos_neg_sentence_counts_length,negative_seed_sentence_count,-0.205769,0.097158,3.418419e-02,0.247181,378,
6,M2_pos_neg_sentence_counts_length,total_word_count,0.743170,0.079334,7.419981e-21,0.247181,378,
7,M3_pos_neg_sentence_shares_volume,const,2.619048,0.070684,1.619526e-300,0.292281,378,
8,M3_pos_neg_sentence_shares_volume,positive_seed_sentence_share,0.268167,0.074814,3.378262e-04,0.292281,378,
9,M3_pos_neg_sentence_shares_volume,negative_seed_sentence_share,-0.189376,0.066546,4.430362e-03,0.292281,378,


,model,variable,coef,std_err,p_value,r2,n,skip_reason
43,M10_term_context_share_interaction,seed_term_context_count,0.858846,0.148242,6.891655e-09,0.335056,378,
41,M10_term_context_share_interaction,negative_seed_term_context_share,-0.281158,0.070738,7.048476e-05,0.335056,378,
42,M10_term_context_share_interaction,term_context_share_interaction,-0.214806,0.090737,1.791635e-02,0.335056,378,
44,M10_term_context_share_interaction,total_word_count,0.197546,0.096791,4.125502e-02,0.335056,378,
40,M10_term_context_share_interaction,positive_seed_term_context_share,0.047274,0.081787,5.632552e-01,0.335056,378,
1,M1_pos_neg_sentence_counts,positive_seed_sentence_count,0.678335,0.121480,2.351748e-08,0.131508,378,
2,M1_pos_neg_sentence_counts,negative_seed_sentence_count,-0.129014,0.121349,2.877071e-01,0.131508,378,
6,M2_pos_neg_sentence_counts_length,total_word_count,0.743170,0.079334,7.419981e-21,0.247181,378,
5,M2_pos_neg_sentence_counts_length,negative_seed_sentence_count,-0.205769,0.097158,3.418419e-02,0.247181,378,
4,M2_pos_neg_sentence_counts_length,positive_seed_sentence_count,0.238031,0.126036,5.894600e-02,0.247181,378,


In [7]:
# High/low combination analysis.
# Within each fiscal year, classify positive and negative shares into high/low groups
# to avoid year-level shifts driving the grouping.

def yearwise_high_low(series):
    median = series.median(skipna=True)
    return np.where(series >= median, "high", "low")

analysis_df["positive_sentence_group"] = (
    analysis_df.groupby("fiscal_year")["positive_seed_sentence_share"].transform(yearwise_high_low)
)
analysis_df["negative_sentence_group"] = (
    analysis_df.groupby("fiscal_year")["negative_seed_sentence_share"].transform(yearwise_high_low)
)
analysis_df["pos_neg_sentence_group"] = (
    "pos_" + analysis_df["positive_sentence_group"] + "__neg_" + analysis_df["negative_sentence_group"]
)

analysis_df["positive_context_group"] = (
    analysis_df.groupby("fiscal_year")["positive_seed_term_context_share"].transform(yearwise_high_low)
)
analysis_df["negative_context_group"] = (
    analysis_df.groupby("fiscal_year")["negative_seed_term_context_share"].transform(yearwise_high_low)
)
analysis_df["pos_neg_context_group"] = (
    "pos_" + analysis_df["positive_context_group"] + "__neg_" + analysis_df["negative_context_group"]
)

sentence_group_summary = (
    analysis_df.groupby("pos_neg_sentence_group")
    .agg(
        n=("esg_grade_num", "count"),
        mean_esg_grade=("esg_grade_num", "mean"),
        median_esg_grade=("esg_grade_num", "median"),
        mean_positive_share=("positive_seed_sentence_share", "mean"),
        mean_negative_share=("negative_seed_sentence_share", "mean"),
    )
    .reset_index()
    .sort_values("mean_esg_grade", ascending=False)
)

context_group_summary = (
    analysis_df.groupby("pos_neg_context_group")
    .agg(
        n=("esg_grade_num", "count"),
        mean_esg_grade=("esg_grade_num", "mean"),
        median_esg_grade=("esg_grade_num", "median"),
        mean_positive_context_share=("positive_seed_term_context_share", "mean"),
        mean_negative_context_share=("negative_seed_term_context_share", "mean"),
    )
    .reset_index()
    .sort_values("mean_esg_grade", ascending=False)
)

print("Sentence positive/negative high-low groups")
display(sentence_group_summary)
print("Term-context positive/negative high-low groups")
display(context_group_summary)

# Non-parametric comparison across the four groups.
def kruskal_group_test(data, group_col, value_col):
    groups = [g[value_col].dropna().values for _, g in data.groupby(group_col) if len(g[value_col].dropna()) > 0]
    if len(groups) < 2:
        return {"group_col": group_col, "statistic": np.nan, "p_value": np.nan}
    stat, p_value = kruskal(*groups)
    return {"group_col": group_col, "statistic": float(stat), "p_value": float(p_value)}

kruskal_df = pd.DataFrame([
    kruskal_group_test(analysis_df, "pos_neg_sentence_group", "esg_grade_num"),
    kruskal_group_test(analysis_df, "pos_neg_context_group", "esg_grade_num"),
])
display(kruskal_df)

Sentence positive/negative high-low groups


,pos_neg_sentence_group,n,mean_esg_grade,median_esg_grade,mean_positive_share,mean_negative_share
1,pos_high__neg_low,84,3.250000,4.0,0.130373,0.013693
0,pos_high__neg_high,105,3.057143,4.0,0.137532,0.044304
3,pos_low__neg_low,105,2.114286,2.0,0.029578,0.009874
2,pos_low__neg_high,84,2.071429,2.0,0.038459,0.050042


Term-context positive/negative high-low groups


,pos_neg_context_group,n,mean_esg_grade,median_esg_grade,mean_positive_context_share,mean_negative_context_share
1,pos_high__neg_low,83,3.168675,4.0,0.099429,0.009105
0,pos_high__neg_high,106,3.141509,4.0,0.106637,0.037506
3,pos_low__neg_low,106,2.235849,2.0,0.020635,0.006642
2,pos_low__neg_high,83,1.891566,2.0,0.026490,0.041754


,group_col,statistic,p_value
0,pos_neg_sentence_group,40.969653,6.636835e-09
1,pos_neg_context_group,45.137817,8.649249e-10


In [8]:
# Year-by-year check: do the signs and patterns hold within each fiscal year?

yearly_rows = []
YEARLY_FEATURES = [
    "positive_seed_sentence_share", "negative_seed_sentence_share", "net_seed_sentence_share",
    "positive_seed_sentence_count", "negative_seed_sentence_count", "net_seed_sentence_count",
    "mean_seed_sentence_sentiment", "positive_seed_term_context_share", "negative_seed_term_context_share",
    "net_seed_term_context_share", "mean_seed_term_context_sentiment",
]
for fiscal_year, year_df in analysis_df.groupby("fiscal_year"):
    for feature in YEARLY_FEATURES:
        n, rho, p_value = spearman_for_feature(year_df, "esg_grade_num", feature)
        yearly_rows.append({
            "fiscal_year": fiscal_year,
            "esg_year": year_df["esg_year"].iloc[0],
            "feature": feature,
            "n": n,
            "spearman_rho": rho,
            "p_value": p_value,
        })

yearly_spearman_df = pd.DataFrame(yearly_rows).sort_values(
    ["fiscal_year", "spearman_rho"], ascending=[True, False]
).reset_index(drop=True)
display(yearly_spearman_df)

YEARLY_MODEL_SPECS = {
    "Y1_sentence_shares_volume": [
        "positive_seed_sentence_share", "negative_seed_sentence_share", "seed_sentence_count", "total_word_count",
    ],
    "Y2_net_sentence_share_volume": [
        "net_seed_sentence_share", "seed_sentence_count", "total_word_count",
    ],
}

yearly_ols_frames = []
for fiscal_year, year_df in analysis_df.groupby("fiscal_year"):
    frame = run_model_table(year_df, YEARLY_MODEL_SPECS)
    frame.insert(0, "fiscal_year", fiscal_year)
    frame.insert(1, "esg_year", year_df["esg_year"].iloc[0])
    yearly_ols_frames.append(frame)

yearly_ols_df = pd.concat(yearly_ols_frames, ignore_index=True)
display(yearly_ols_df)

,fiscal_year,esg_year,feature,n,spearman_rho,p_value
0,2022,2023,positive_seed_sentence_count,126,0.577051,1.523290e-12
1,2022,2023,net_seed_sentence_count,126,0.511875,9.012921e-10
2,2022,2023,positive_seed_sentence_share,126,0.365228,2.610595e-05
3,2022,2023,positive_seed_term_context_share,126,0.348372,6.402616e-05
4,2022,2023,net_seed_sentence_share,126,0.342921,8.466787e-05
5,2022,2023,mean_seed_sentence_sentiment,126,0.334571,1.286234e-04
6,2022,2023,net_seed_term_context_share,126,0.316071,3.116537e-04
7,2022,2023,mean_seed_term_context_sentiment,126,0.306692,4.778663e-04
8,2022,2023,negative_seed_sentence_count,126,0.304543,5.260012e-04
9,2022,2023,negative_seed_sentence_share,126,0.052139,5.620361e-01


,fiscal_year,esg_year,model,variable,coef,std_err,p_value,r2,n,skip_reason
0,2022,2023,Y1_sentence_shares_volume,const,2.555556,0.124734,2.753780e-93,0.332969,126,
1,2022,2023,Y1_sentence_shares_volume,positive_seed_sentence_share,0.245823,0.132709,6.397675e-02,0.332969,126,
2,2022,2023,Y1_sentence_shares_volume,negative_seed_sentence_share,-0.203952,0.121681,9.371396e-02,0.332969,126,
3,2022,2023,Y1_sentence_shares_volume,seed_sentence_count,0.478416,0.292514,1.019380e-01,0.332969,126,
4,2022,2023,Y1_sentence_shares_volume,total_word_count,0.391301,0.183827,3.328393e-02,0.332969,126,
5,2022,2023,Y2_net_sentence_share_volume,const,2.555556,0.124422,9.556222e-94,0.329021,126,
6,2022,2023,Y2_net_sentence_share_volume,net_seed_sentence_share,0.284947,0.128081,2.609788e-02,0.329021,126,
7,2022,2023,Y2_net_sentence_share_volume,seed_sentence_count,0.424594,0.310292,1.711970e-01,0.329021,126,
8,2022,2023,Y2_net_sentence_share_volume,total_word_count,0.404912,0.185800,2.931025e-02,0.329021,126,
9,2023,2024,Y1_sentence_shares_volume,const,2.674603,0.126003,5.437572e-100,0.308572,126,


In [9]:
# Year-over-year change analysis.
# This directly tests whether changes in positive/negative seed contexts are associated
# with changes in ESG grade. With 3 fiscal years, each firm can contribute up to 2 transitions.

CHANGE_BASE_COLS = [
    "esg_grade_num",
    "positive_seed_sentence_count",
    "negative_seed_sentence_count",
    "net_seed_sentence_count",
    "positive_seed_sentence_share",
    "negative_seed_sentence_share",
    "net_seed_sentence_share",
    "mean_seed_sentence_sentiment",
    "positive_seed_term_context_count",
    "negative_seed_term_context_count",
    "net_seed_term_context_count",
    "positive_seed_term_context_share",
    "negative_seed_term_context_share",
    "net_seed_term_context_share",
    "mean_seed_term_context_sentiment",
    "seed_sentence_count",
    "seed_term_context_count",
    "unique_seed_terms_matched",
    "total_word_count",
]

panel_df = analysis_df.sort_values(["stock_code", "fiscal_year"]).copy()
for col in CHANGE_BASE_COLS:
    if col not in panel_df.columns:
        continue
    panel_df[f"lag_{col}"] = panel_df.groupby("stock_code")[col].shift(1)
    panel_df[f"delta_{col}"] = panel_df[col] - panel_df[f"lag_{col}"]

panel_df["lag_fiscal_year"] = panel_df.groupby("stock_code")["fiscal_year"].shift(1)
panel_df["year_gap"] = panel_df["fiscal_year"] - panel_df["lag_fiscal_year"]
change_df = panel_df[panel_df["year_gap"].eq(1)].copy()
change_df["grade_change_group"] = np.select(
    [change_df["delta_esg_grade_num"].gt(0), change_df["delta_esg_grade_num"].lt(0)],
    ["increase", "decrease"],
    default="no_change",
)

print("change rows:", len(change_df))
display(change_df.groupby(["fiscal_year", "esg_year", "grade_change_group"]).size().rename("n").reset_index())
display(change_df[[
    "stock_code", "company_name", "fiscal_year", "esg_year", "esg_grade_num", "lag_esg_grade_num", "delta_esg_grade_num",
    "delta_positive_seed_sentence_share", "delta_negative_seed_sentence_share", "delta_net_seed_sentence_share",
    "delta_positive_seed_sentence_count", "delta_negative_seed_sentence_count", "delta_total_word_count",
    "grade_change_group",
]].head(20))

change rows: 252


,fiscal_year,esg_year,grade_change_group,n
0,2023,2024,decrease,23
1,2023,2024,increase,34
2,2023,2024,no_change,69
3,2024,2025,decrease,30
4,2024,2025,increase,23
5,2024,2025,no_change,73


,stock_code,company_name,fiscal_year,esg_year,esg_grade_num,lag_esg_grade_num,delta_esg_grade_num,delta_positive_seed_sentence_share,delta_negative_seed_sentence_share,delta_net_seed_sentence_share,delta_positive_seed_sentence_count,delta_negative_seed_sentence_count,delta_total_word_count,grade_change_group
1,000020,동화약품,2023,2024,1,1.0,0.0,0.000000,-0.028571,0.028571,0.0,-1.0,257.0,no_change
2,000020,동화약품,2024,2025,1,1.0,0.0,-0.003089,-0.028571,0.025483,0.0,-1.0,32.0,no_change
4,000040,KR모터스,2023,2024,0,0.0,0.0,-0.090909,0.090909,-0.181818,-1.0,1.0,687.0,no_change
5,000040,KR모터스,2024,2025,0,0.0,0.0,0.066667,-0.090909,0.157576,1.0,-1.0,-641.0,no_change
7,000050,경방,2023,2024,1,1.0,0.0,-0.027574,0.000000,-0.027574,-1.0,0.0,326.0,no_change
8,000050,경방,2024,2025,1,1.0,0.0,0.000000,0.031250,-0.031250,0.0,1.0,-57.0,no_change
10,000070,삼양홀딩스,2023,2024,4,3.0,1.0,0.109463,-0.019949,0.129412,10.0,-1.0,546.0,increase
11,000070,삼양홀딩스,2024,2025,3,4.0,-1.0,0.032244,-0.011184,0.043428,2.0,-1.0,-85.0,decrease
13,000080,하이트진로,2023,2024,3,2.0,1.0,-0.031884,0.023188,-0.055072,-3.0,2.0,-653.0,increase
14,000080,하이트진로,2024,2025,2,3.0,-1.0,0.014286,-0.054762,0.069048,1.0,-5.0,134.0,decrease


In [10]:
# Spearman correlations between sentiment changes and ESG-grade changes.
# Overall and by target fiscal_year/esg_year transition.

DELTA_FEATURES = [
    "delta_positive_seed_sentence_count",
    "delta_negative_seed_sentence_count",
    "delta_net_seed_sentence_count",
    "delta_positive_seed_sentence_share",
    "delta_negative_seed_sentence_share",
    "delta_net_seed_sentence_share",
    "delta_mean_seed_sentence_sentiment",
    "delta_positive_seed_term_context_count",
    "delta_negative_seed_term_context_count",
    "delta_net_seed_term_context_count",
    "delta_positive_seed_term_context_share",
    "delta_negative_seed_term_context_share",
    "delta_net_seed_term_context_share",
    "delta_mean_seed_term_context_sentiment",
    "delta_seed_sentence_count",
    "delta_seed_term_context_count",
    "delta_unique_seed_terms_matched",
    "delta_total_word_count",
]

change_spearman_rows = []
for feature in DELTA_FEATURES:
    if feature not in change_df.columns:
        continue
    n, rho, p_value = spearman_for_feature(change_df, "delta_esg_grade_num", feature)
    change_spearman_rows.append({
        "scope": "overall",
        "fiscal_year": "all",
        "esg_year": "all",
        "feature": feature,
        "n": n,
        "spearman_rho": rho,
        "p_value": p_value,
    })

for (fiscal_year, esg_year), year_df in change_df.groupby(["fiscal_year", "esg_year"]):
    for feature in DELTA_FEATURES:
        if feature not in year_df.columns:
            continue
        n, rho, p_value = spearman_for_feature(year_df, "delta_esg_grade_num", feature)
        change_spearman_rows.append({
            "scope": "year_transition",
            "fiscal_year": fiscal_year,
            "esg_year": esg_year,
            "feature": feature,
            "n": n,
            "spearman_rho": rho,
            "p_value": p_value,
        })

change_spearman_df = pd.DataFrame(change_spearman_rows).sort_values(
    ["scope", "fiscal_year", "spearman_rho"], ascending=[True, True, False]
).reset_index(drop=True)
display(change_spearman_df)

,scope,fiscal_year,esg_year,feature,n,spearman_rho,p_value
0,overall,all,all,delta_seed_term_context_count,252,0.150234,0.017005
1,overall,all,all,delta_seed_sentence_count,252,0.145694,0.020685
2,overall,all,all,delta_total_word_count,252,0.126883,0.044185
3,overall,all,all,delta_negative_seed_sentence_count,252,0.073290,0.246369
4,overall,all,all,delta_negative_seed_term_context_count,252,0.062506,0.323009
5,overall,all,all,delta_negative_seed_term_context_share,252,0.051677,0.414032
6,overall,all,all,delta_negative_seed_sentence_share,252,0.029698,0.638923
7,overall,all,all,delta_mean_seed_sentence_sentiment,252,0.026705,0.673105
8,overall,all,all,delta_net_seed_sentence_count,252,0.019007,0.763979
9,overall,all,all,delta_positive_seed_term_context_count,252,0.017823,0.778288


In [11]:
# Change regressions.
# Positive and negative changes are included together, with volume changes controlled.

CHANGE_MODEL_SPECS = {
    "D1_delta_pos_neg_sentence_counts": [
        "delta_positive_seed_sentence_count", "delta_negative_seed_sentence_count",
    ],
    "D2_delta_pos_neg_sentence_counts_volume": [
        "delta_positive_seed_sentence_count", "delta_negative_seed_sentence_count", "delta_total_word_count",
    ],
    "D3_delta_pos_neg_sentence_shares_volume": [
        "delta_positive_seed_sentence_share", "delta_negative_seed_sentence_share", "delta_seed_sentence_count", "delta_total_word_count",
    ],
    "D4_delta_net_sentence_share_volume": [
        "delta_net_seed_sentence_share", "delta_seed_sentence_count", "delta_total_word_count",
    ],
    "D5_delta_pos_neg_context_shares_volume": [
        "delta_positive_seed_term_context_share", "delta_negative_seed_term_context_share", "delta_seed_term_context_count", "delta_total_word_count",
    ],
    "D6_delta_net_context_share_volume": [
        "delta_net_seed_term_context_share", "delta_seed_term_context_count", "delta_total_word_count",
    ],
}

change_ols_df = run_model_table(change_df, CHANGE_MODEL_SPECS, y_col="delta_esg_grade_num")
display(change_ols_df)

yearly_change_ols_frames = []
for (fiscal_year, esg_year), year_df in change_df.groupby(["fiscal_year", "esg_year"]):
    frame = run_model_table(year_df, CHANGE_MODEL_SPECS, y_col="delta_esg_grade_num")
    frame.insert(0, "fiscal_year", fiscal_year)
    frame.insert(1, "esg_year", esg_year)
    yearly_change_ols_frames.append(frame)

yearly_change_ols_df = pd.concat(yearly_change_ols_frames, ignore_index=True)
display(yearly_change_ols_df)

,model,variable,coef,std_err,p_value,r2,n,skip_reason
0,D1_delta_pos_neg_sentence_counts,const,0.035714,0.049095,0.466947,0.002955,252,
1,D1_delta_pos_neg_sentence_counts,delta_positive_seed_sentence_count,0.002456,0.065603,0.970134,0.002955,252,
2,D1_delta_pos_neg_sentence_counts,delta_negative_seed_sentence_count,0.041908,0.058564,0.474243,0.002955,252,
3,D2_delta_pos_neg_sentence_counts_volume,const,0.035714,0.049340,0.469162,0.005506,252,
4,D2_delta_pos_neg_sentence_counts_volume,delta_positive_seed_sentence_count,-0.018145,0.072740,0.803011,0.005506,252,
5,D2_delta_pos_neg_sentence_counts_volume,delta_negative_seed_sentence_count,0.033625,0.061528,0.584725,0.005506,252,
6,D2_delta_pos_neg_sentence_counts_volume,delta_total_word_count,0.044348,0.070526,0.529466,0.005506,252,
7,D3_delta_pos_neg_sentence_shares_volume,const,0.035714,0.049547,0.471020,0.005374,252,
8,D3_delta_pos_neg_sentence_shares_volume,delta_positive_seed_sentence_share,0.037990,0.049744,0.445034,0.005374,252,
9,D3_delta_pos_neg_sentence_shares_volume,delta_negative_seed_sentence_share,0.020898,0.055452,0.706280,0.005374,252,


,fiscal_year,esg_year,model,variable,coef,std_err,p_value,r2,n,skip_reason
0,2023,2024,D1_delta_pos_neg_sentence_counts,const,0.119048,0.070808,0.092709,0.001874,126,
1,2023,2024,D1_delta_pos_neg_sentence_counts,delta_positive_seed_sentence_count,0.028820,0.103620,0.780910,0.001874,126,
2,2023,2024,D1_delta_pos_neg_sentence_counts,delta_negative_seed_sentence_count,0.018948,0.118826,0.873306,0.001874,126,
3,2023,2024,D2_delta_pos_neg_sentence_counts_volume,const,0.119048,0.071599,0.096374,0.004113,126,
4,2023,2024,D2_delta_pos_neg_sentence_counts_volume,delta_positive_seed_sentence_count,0.005028,0.111817,0.964133,0.004113,126,
5,2023,2024,D2_delta_pos_neg_sentence_counts_volume,delta_negative_seed_sentence_count,0.006165,0.146010,0.966322,0.004113,126,
6,2023,2024,D2_delta_pos_neg_sentence_counts_volume,delta_total_word_count,0.044437,0.082475,0.590024,0.004113,126,
7,2023,2024,D3_delta_pos_neg_sentence_shares_volume,const,0.119048,0.070778,0.092572,0.014169,126,
8,2023,2024,D3_delta_pos_neg_sentence_shares_volume,delta_positive_seed_sentence_share,0.043750,0.078508,0.577342,0.014169,126,
9,2023,2024,D3_delta_pos_neg_sentence_shares_volume,delta_negative_seed_sentence_share,-0.052595,0.076913,0.494091,0.014169,126,


In [12]:
# Group comparison: do firms with grade increases show different positive/negative changes?

CHANGE_GROUP_COLS = [
    "delta_positive_seed_sentence_share",
    "delta_negative_seed_sentence_share",
    "delta_net_seed_sentence_share",
    "delta_positive_seed_sentence_count",
    "delta_negative_seed_sentence_count",
    "delta_net_seed_sentence_count",
    "delta_mean_seed_sentence_sentiment",
]

change_group_summary = (
    change_df.groupby("grade_change_group")[CHANGE_GROUP_COLS]
    .agg(["count", "mean", "median", "std"])
)
display(change_group_summary)

change_group_tests = []
for feature in CHANGE_GROUP_COLS:
    groups = [g[feature].dropna().values for _, g in change_df.groupby("grade_change_group") if len(g[feature].dropna()) > 0]
    if len(groups) < 2:
        stat, p_value = np.nan, np.nan
    else:
        stat, p_value = kruskal(*groups)
    change_group_tests.append({"feature": feature, "test": "Kruskal-Wallis", "statistic": stat, "p_value": p_value})

change_group_test_df = pd.DataFrame(change_group_tests).sort_values("p_value")
display(change_group_test_df)

delta_positive_seed_sentence_share                      \
                                                count      mean    median   
grade_change_group                                                          
decrease                                           53 -0.004860 -0.001663   
increase                                           57  0.000477 -0.001812   
no_change                                         142 -0.000305  0.000000   

                             delta_negative_seed_sentence_share            \
                         std                              count      mean   
grade_change_group                                                          
decrease            0.029372                                 53 -0.002535   
increase            0.037576                                 57 -0.000091   
no_change           0.030976                                142  0.001959   

                                    delta_net_seed_sentence_share            \
                   median       std                         count      mean   
grade_change_group                                                            
decrease              0.0  0.019680                            53 -0.002325   
increase              0.0  0.018635                            57  0.000568   
no_change             0.0  0.021037                           142 -0.002264   

                                       delta_positive_seed_sentence_count  \
                      median       std                              count   
grade_change_group                                                          
decrease           -0.001663  0.038814                                 53   
increase           -0.003810  0.042411                                 57   
no_change           0.000000  0.042973                                142   

                                               \
                        mean median       std   
grade_change_group                              
decrease            0.037736    0.0  6.406014   
increase            0.000000    0.0  6.056284   
no_change           0.309859    0.0  2.775739   

                   delta_negative_seed_sentence_count                   \
                                                count      mean median   
grade_change_group                                                       
decrease                                           53 -0.490566    0.0   
increase                                           57  0.000000    0.0   
no_change                                         142  0.239437    0.0   

                             delta_net_seed_sentence_count                   \
                         std                         count      mean median   
grade_change_group                                                            
decrease            2.784729                            53  0.528302    0.0   
increase            1.782855                            57  0.000000    0.0   
no_change           1.423857                           142  0.070423    0.0   

                             delta_mean_seed_sentence_sentiment            \
                         std                              count      mean   
grade_change_group                                                          
decrease            7.645018                                 53 -0.003580   
increase            6.123724                                 57  0.000834   
no_change           3.172685                                142 -0.001664   

                                        
                      median       std  
grade_change_group                      
decrease           -0.001442  0.035134  
increase           -0.001174  0.038192  
no_change           0.000000  0.035656

,feature,test,statistic,p_value
4,delta_negative_seed_sentence_count,Kruskal-Wallis,4.121110,0.127383
1,delta_negative_seed_sentence_share,Kruskal-Wallis,1.793027,0.407990
3,delta_positive_seed_sentence_count,Kruskal-Wallis,1.753213,0.416193
0,delta_positive_seed_sentence_share,Kruskal-Wallis,0.760481,0.683697
6,delta_mean_seed_sentence_sentiment,Kruskal-Wallis,0.404201,0.817013
5,delta_net_seed_sentence_count,Kruskal-Wallis,0.393057,0.821578
2,delta_net_seed_sentence_share,Kruskal-Wallis,0.180435,0.913733


In [13]:
OUTPUT_CHANGE_PATH = FINAL_DIR / "v_2_2_seed_context_sentiment_controls_change.csv"
OUTPUT_CHANGE_SPEARMAN_PATH = FINAL_DIR / "v_2_2_seed_context_sentiment_controls_change_spearman.csv"
OUTPUT_CHANGE_OLS_PATH = FINAL_DIR / "v_2_2_seed_context_sentiment_controls_change_ols.csv"
OUTPUT_YEARLY_CHANGE_OLS_PATH = FINAL_DIR / "v_2_2_seed_context_sentiment_controls_yearly_change_ols.csv"

change_df.to_csv(OUTPUT_CHANGE_PATH, index=False, encoding="utf-8-sig")
change_spearman_df.to_csv(OUTPUT_CHANGE_SPEARMAN_PATH, index=False, encoding="utf-8-sig")
change_ols_df.to_csv(OUTPUT_CHANGE_OLS_PATH, index=False, encoding="utf-8-sig")
yearly_change_ols_df.to_csv(OUTPUT_YEARLY_CHANGE_OLS_PATH, index=False, encoding="utf-8-sig")

print("saved change:", OUTPUT_CHANGE_PATH, change_df.shape)
print("saved change spearman:", OUTPUT_CHANGE_SPEARMAN_PATH, change_spearman_df.shape)
print("saved change ols:", OUTPUT_CHANGE_OLS_PATH, change_ols_df.shape)
print("saved yearly change ols:", OUTPUT_YEARLY_CHANGE_OLS_PATH, yearly_change_ols_df.shape)

saved change: /content/drive/MyDrive/UD_26/final/v_2_2_seed_context_sentiment_controls_change.csv (252, 102)
saved change spearman: /content/drive/MyDrive/UD_26/final/v_2_2_seed_context_sentiment_controls_change_spearman.csv (54, 7)
saved change ols: /content/drive/MyDrive/UD_26/final/v_2_2_seed_context_sentiment_controls_change_ols.csv (25, 8)
saved yearly change ols: /content/drive/MyDrive/UD_26/final/v_2_2_seed_context_sentiment_controls_yearly_change_ols.csv (50, 10)


In [14]:
OUTPUT_ANALYSIS_PATH = FINAL_DIR / "v_2_2_seed_context_sentiment_controls_analysis.csv"
OUTPUT_SPEARMAN_PATH = FINAL_DIR / "v_2_2_seed_context_sentiment_controls_spearman.csv"
OUTPUT_OLS_PATH = FINAL_DIR / "v_2_2_seed_context_sentiment_controls_ols.csv"
OUTPUT_YEARLY_SPEARMAN_PATH = FINAL_DIR / "v_2_2_seed_context_sentiment_controls_yearly_spearman.csv"

analysis_df.to_csv(OUTPUT_ANALYSIS_PATH, index=False, encoding="utf-8-sig")
spearman_df.to_csv(OUTPUT_SPEARMAN_PATH, index=False, encoding="utf-8-sig")
ols_df.to_csv(OUTPUT_OLS_PATH, index=False, encoding="utf-8-sig")
yearly_spearman_df.to_csv(OUTPUT_YEARLY_SPEARMAN_PATH, index=False, encoding="utf-8-sig")

print("saved analysis:", OUTPUT_ANALYSIS_PATH, analysis_df.shape)
print("saved spearman:", OUTPUT_SPEARMAN_PATH, spearman_df.shape)
print("saved ols:", OUTPUT_OLS_PATH, ols_df.shape)
print("saved yearly spearman:", OUTPUT_YEARLY_SPEARMAN_PATH, yearly_spearman_df.shape)

saved analysis: /content/drive/MyDrive/UD_26/final/v_2_2_seed_context_sentiment_controls_analysis.csv (378, 61)
saved spearman: /content/drive/MyDrive/UD_26/final/v_2_2_seed_context_sentiment_controls_spearman.csv (19, 4)
saved ols: /content/drive/MyDrive/UD_26/final/v_2_2_seed_context_sentiment_controls_ols.csv (45, 8)
saved yearly spearman: /content/drive/MyDrive/UD_26/final/v_2_2_seed_context_sentiment_controls_yearly_spearman.csv (33, 6)
